In [1]:
# Import the necessary libraries and load the data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("/Users/serhatguldogan/Library/CloudStorage/OneDrive-KocUniversitesi/Project_/RAW DATA/BTC_1sec.csv")


In [2]:
df.head()

,Unnamed: 0,system_time,midpoint,spread,buys,sells,bids_distance_0,bids_distance_1,bids_distance_2,bids_distance_3,...,asks_market_notional_5,asks_market_notional_6,asks_market_notional_7,asks_market_notional_8,asks_market_notional_9,asks_market_notional_10,asks_market_notional_11,asks_market_notional_12,asks_market_notional_13,asks_market_notional_14
0,0,2021-04-07 11:32:42.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2021-04-07 11:32:43.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2021-04-07 11:32:44.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2021-04-07 11:32:45.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2021-04-07 11:32:46.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
prices = df["midpoint"]

In [5]:
# Count the transitions. We have:
# 1. Price up - Price up
# 2. Price up- Price down
# 3. Price down - Price up
# 4. Price down - Price down
p_uu = 0
p_ud = 0
p_du=0
p_dd=0
for i in range(np.size(prices)-2):
    if prices[i+1] > prices[i] and prices[i+2] > prices[i+1]:
        p_uu +=1
    if prices[i+1] > prices[i] and prices[i+2] < prices[i+1]:
        p_ud +=1
    
        
    if prices[i+1] < prices[i]:
        if prices [i+2] > prices[i+1]:
            p_du +=1
        if prices[i+2] < prices[i+1]:
            p_dd +=1
    




In [ ]:
# Normalise.
P_uu = (p_uu) / (p_uu + p_ud)
P_ud = 1- P_uu
P_du = (p_du) / (p_du + p_dd)
P_dd = 1 - P_du

In [9]:
# Then get the transition probability matrix for our markov chain.
M= np.array(([[P_uu,P_ud],[P_du,P_dd]]))
print(M)

[[0.59758503 0.40241497]
 [0.44562421 0.55437579]]


Below is the solution for the eigenvector problem. In the end, we get the stationary probability vector.

In [10]:
# Solve the equation vP = v where P is the matrix "mat" (i.e., find the stationary distribution)

# To solve vP = v, equivalently solve v(P - I) = 0 with constraint sum(v) = 1
# We transpose so that v is a row vector: v @ P = v <=> (P.T - I.T) v^T = 0
from numpy.linalg import svd

P = M
size = P.shape[0]
A = P.T - np.eye(size)  # (P^T - I) so that nullspace is stationary v^T

# Append the normalization equation sum(v) = 1
A = np.vstack([A, np.ones(size)])
b = np.zeros(size + 1)
b[-1] = 1

# Solve using least squares (the last row enforces sum(v) = 1)
v, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

# v is the stationary distribution
v = v.real  # ensure no tiny imaginary part from solver

v = np.array(v) # Display the stationary distribution vector

Our stationary probability vector. First component is for $+\delta$, second is for $-\delta$

In [13]:
print(v)

[0.52547597 0.47452403]


Calculate the asymptotic mean for $X_k$'s, i.e , $a^* = \sum \pi_k X_k$.

In [12]:

as_mean = 0.5*(v[0]-v[1])
as_mean
    
    

0.02547596738898039